# ⚠️ SUPERSEDED IN PART — read this before using the recommendation below

**The `N` axis of this study is mis-designed, and its `(M, L, N)` recommendation should not be used as-is.**

Every sweep below refines one truncation while holding the others fixed. For `M` and `L` that is fine. For `N` it is not: the number of pressure directions the model can respond to at all is set by the *bath and droplet* truncations and the contact angle — the resolvable-rank (bandwidth) law `n_*(M, L, θ_c)`, now pinned in CI by `test/test_rank_law.jl` and documented in `derivations/DIAGNOSTICS-NOTATION.md`. Refining `N` past `n_*` at fixed `M, L` adds trial directions the test space cannot see, so the `N`-sweep **cannot** converge by construction; its non-convergence here is an artifact of the experiment, not a property of the model.

Measured at `N = 12`, `θ_c = 0.3`, same basis, nothing else changed:

| `M = L` | `cond(J_12)` |
|---|---|
| 80 | 5.0e10 |
| 320 | 7.2e1 |
| 640 | 3.2e1 |

Nine orders of magnitude from refining `M, L` alone. Correspondingly, over a full impact at `M = L = 60`, the fraction of contact steps with negative (unphysical) pressure is 2% at `N = 3`, 2% at `N = 6`, and **90%** at `N = 12` — because `n_*(L=60, θ_c≈0.3) ≈ 11` puts `N = 12` far outside the budget.

**What still stands:** the `M` and `L` sweeps, and the observation that decoupled sweeps converge earlier than the joint `M = L` sweep (the confound the design doc's withdrawn `M = L`-only study warned about).

**What to do instead:** provision `N` against the budget (`N + 1 ≲ 0.6 n_*`) and refine `M`, `L`, `N` *jointly*. A replacement joint sweep is pending; until it exists, no `(M, L, N)` recommendation in this notebook has been validated end-to-end through a live impact.


# Convergence study: M, L, N, and a joint cross-check

`derivations/provenance.tex` records this as open: whether the pressure moments $c_m$, $b_l$,
$f$ -- not just the integrated metrics (contact time, CoR) -- actually converge as the bath
truncation `M`, droplet truncation `L`, and pressure order `N` are refined, jointly, has never
been checked. Only `N`-insensitivity of the integrated metrics was checked before (the
tutorial notebook's own N-sweep). This notebook checks the moments directly, decouples `M`
from `L` rather than assuming `M=L` (the design doc's own history has a withdrawn scaling law
that made exactly that assumption and missed a real effect because of it), and ends with an
explicit, tolerance-based verdict rather than "not established."

**A note on scope, discovered while building this.** The plan going in included a fourth
sweep axis: the Newton solve's "spectral-filter cutoff". It doesn't exist -- the actual
solver (`src/newton.jl`) does Levenberg-Marquardt-damped least squares, unconditionally, with
no truncation of any kind; the design doc's description of an SVD-filtered solve was stale and
has been corrected (`derivations/provenance.tex`, corrections). What *is* real and directly
relevant here: the Newton Jacobian's condition number is order-unity at the production `N=3`
but grows explosively with `N` alone (measured: $3.2$ at $N=3$, $4.4\times10^6$ at $N=10$,
$2.9\times10^{17}$ at $N=15$, at fixed $M=L=80$). The `N`-sweep below tracks `cond(J)`
alongside the physics, so a run that stops changing because it converged is distinguishable
from one that stopped changing because the solver is quietly regularization-dominated.

**Tolerance.** A quantity is called converged once its relative change between successive
refinements drops below $1\%$ *and stays there* for the next refinement too -- one lucky pair
is not enough, following the same standard `derivations/`'s own audits use elsewhere.

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using SpectralKM, Printf, SpecialFunctions, Base64, ForwardDiff, LinearAlgebra
using SpectralKM: step_affine, contact_quad, residual, SimHistory

  Activating 

project at `~/Documents/Github/1pkm-drop-onto-bath`


## A minimal SVG toolkit

Same line-plot primitive the tutorial notebook uses. No plotting dependency.

In [2]:
function svgplot(series; width=680, height=300, xlabel="", ylabel="", title="",
                 xlim=nothing, ylim=nothing)
    pad = 52
    xs = vcat((s.x for s in series)...); ys = vcat((s.y for s in series)...)
    x0, x1 = xlim === nothing ? (minimum(xs), maximum(xs)) : xlim
    y0, y1 = ylim === nothing ? (minimum(ys), maximum(ys)) : ylim
    x1 == x0 && (x1 = x0 + 1); y1 == y0 && (y1 = y0 + 1)
    sx = v -> pad + (v - x0) / (x1 - x0) * (width - 1.6pad)
    sy = v -> height - pad - (v - y0) / (y1 - y0) * (height - 1.7pad)
    io = IOBuffer()
    print(io, "<svg xmlns=\'http://www.w3.org/2000/svg\' width=\'$width\' height=\'$height\' font-family=\'sans-serif\' font-size=\'12\'>")
    print(io, "<rect width=\'$width\' height=\'$height\' fill=\'white\'/>")
    for (frac) in 0:0.25:1
        xv = x0 + frac * (x1 - x0); yv = y0 + frac * (y1 - y0)
        print(io, "<line x1=\'$(sx(xv))\' y1=\'$(height-pad)\' x2=\'$(sx(xv))\' y2=\'$(height-pad+5)\' stroke=\'black\'/>")
        print(io, "<text x=\'$(sx(xv))\' y=\'$(height-pad+18)\' text-anchor=\'middle\'>$(round(xv,sigdigits=3))</text>")
        print(io, "<line x1=\'$pad\' y1=\'$(sy(yv))\' x2=\'$(pad-5)\' y2=\'$(sy(yv))\' stroke=\'black\'/>")
        print(io, "<text x=\'$(pad-9)\' y=\'$(sy(yv)+4)\' text-anchor=\'end\'>$(round(yv,sigdigits=3))</text>")
    end
    print(io, "<line x1=\'$pad\' y1=\'$(height-pad)\' x2=\'$(width-0.6pad)\' y2=\'$(height-pad)\' stroke=\'black\'/>")
    print(io, "<line x1=\'$pad\' y1=\'$(height-pad)\' x2=\'$pad\' y2=\'$(pad*0.5)\' stroke=\'black\'/>")
    for s in series
        pts = join(("$(sx(s.x[i])),$(sy(s.y[i]))" for i in eachindex(s.x)), " ")
        w = get(s, :width, 1.8)
        print(io, "<polyline points=\'$pts\' fill=\'none\' stroke=\'$(s.color)\' stroke-width=\'$w\'/>")
        for i in eachindex(s.x)
            print(io, "<circle cx=\'$(sx(s.x[i]))\' cy=\'$(sy(s.y[i]))\' r=\'3\' fill=\'$(s.color)\'/>")
        end
    end
    for (i, s) in enumerate(series)
        haskey(s, :label) || continue
        print(io, "<line x1=\'$(width-190)\' y1=\'$(pad*0.5+14i)\' x2=\'$(width-165)\' y2=\'$(pad*0.5+14i)\' stroke=\'$(s.color)\' stroke-width=\'2.4\'/>")
        print(io, "<text x=\'$(width-159)\' y=\'$(pad*0.5+14i+4)\'>$(s.label)</text>")
    end
    print(io, "<text x=\'$(width/2)\' y=\'$(height-8)\' text-anchor=\'middle\'>$xlabel</text>")
    print(io, "<text x=\'14\' y=\'$(height/2)\' text-anchor=\'middle\' transform=\'rotate(-90 14 $(height/2))\'>$ylabel</text>")
    print(io, "<text x=\'$(width/2)\' y=\'18\' text-anchor=\'middle\' font-size=\'14\'>$title</text></svg>")
    return HTML(String(take!(io)))
end

struct HTML s::String end
Base.show(io::IO, ::MIME"text/html", h::HTML) = print(io, h.s)
println("ready")

ready


## The sweep runner

One impact, at a given `(M, L, N)`, reduced to: the integrated metrics, two representative
pressure moments (`c_1`, the first non-piston bath mode; `b_2`, the first real droplet mode)
sampled at the instant of peak penetration -- the same instant across every run, so the
comparison is meaningful -- and `cond(J)` of the Newton Jacobian at that same instant, as the
solver-reliability check motivated above. `t_end=25` throughout (raised from the tutorial's
`14`: this sweep pushes `M`, `L` well past production values, and a longer horizon avoids
truncating a slower rebound at those settings).

In [3]:
function run_case(; M, L, N, t_end=25.0, We=1.0958, Bo=0.017, Oh=0.006, b=6.0, h0=3.0, nq=nothing, wall=:free)
    nq_ = nq === nothing ? max(40, 10 * N) : nq
    p = Params(We=We, Bo=Bo, Oh=Oh, M=M, L=L, N=N, b=b, h0=h0, nq=nq_, wall=wall)
    levels, diag, phases = run_simulation(p; t_end=t_end, dt_init=1e-3)
    isempty(diag) && return nothing
    times = [l.t for l in levels]
    i = argmin(d.z for d in diag)
    dpeak = diag[i]
    li = findfirst(l -> l.t == dpeak.t, levels)
    kappa, alpha, lambda, gam, kappa_cm, mu = step_affine(SimHistory(levels[li-2], levels[li-1]), dpeak.dt, p)
    xc = cos(dpeak.theta_c)
    q = contact_quad(xc, p)
    R = chat -> residual(chat, xc, q, kappa, alpha, lambda, gam, kappa_cm, mu, p)
    J = ForwardDiff.jacobian(R, levels[li].X[1:N+1])
    return (; M, L, N,
            primary=primary_contact_time(times, phases),
            cor=coefficient_of_restitution(times, levels, phases),
            depth=max_penetration_depth(levels, p.L),
            rc_max=sin(maximum(d.theta_c for d in diag)),
            c1=dpeak.cm[2], b2=dpeak.bl[3], fpeak=dpeak.f,
            condJ=cond(J), nsteps=length(levels))
end

"""Relative change (%) of `getter(r)` between successive entries of `results`, ordered by
the swept parameter -- the tolerance check operates on this."""
function reldev(results, getter)
    vals = [getter(r) for r in results]
    [i == 1 ? NaN : 100 * abs(vals[i] - vals[i-1]) / abs(vals[i-1]) for i in eachindex(vals)]
end

function verdict(name, results, getter; tol=1.0)
    d = reldev(results, getter)
    ok = [i >= 3 && d[i] < tol && d[i-1] < tol for i in eachindex(d)]
    idx = findfirst(ok)
    if idx === nothing
        println("  $name: NOT converged to $(tol)% within the range tested")
    else
        v = getter(results[idx-1])
        @printf("  %s: converged by index %d (value %.5g), consecutive deviations %.3g%%, %.3g%%\n",
                name, idx - 1, v, d[idx-1], d[idx])
    end
end
println("ready")

ready


## `M`: bath (Fourier--Bessel) modes

`L=120`, `N=3` held generously fixed so the bath truncation is isolated.

In [4]:
Ms = (20, 40, 60, 90, 120, 180)
results_M = [run_case(M=m, L=120, N=3) for m in Ms]
display(svgplot([(x=collect(Ms[2:end]), y=reldev(results_M, r -> r.primary)[2:end], color="#1f77b4", label="primary_contact_time"),
                  (x=collect(Ms[2:end]), y=reldev(results_M, r -> r.cor)[2:end], color="#d62728", label="CoR")];
                 xlabel="M", ylabel="% change from previous M", title="Integrated metrics vs M"))
display(svgplot([(x=collect(Ms[2:end]), y=reldev(results_M, r -> r.c1)[2:end], color="#2ca02c", label="c_1 (bath moment)"),
                  (x=collect(Ms[2:end]), y=reldev(results_M, r -> r.b2)[2:end], color="#9467bd", label="b_2 (droplet moment)")];
                 xlabel="M", ylabel="% change from previous M", title="Pressure moments vs M"))
println("Verdicts (M-sweep):")
verdict("primary_contact_time", results_M, r -> r.primary)
verdict("CoR", results_M, r -> r.cor)
verdict("c_1", results_M, r -> r.c1)
verdict("b_2", results_M, r -> r.b2)

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>40.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.0</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>75.0</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>0.139</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>110.0</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>0.277</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>145.0</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>0.416</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>180.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>0.554</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,36.39999999999998 137.25714285714287,248.0 265.1428571428571,248.0 393.0285714285714,248.0 648.8,248.0' fill='none' stroke='#1f77b4' stroke-width='1.8'/><circle cx='52.0' cy='36.39999999999998' r='3' fill='#1f77b4'/><circle cx='137.25714285714287' cy='248.0' r='3' fill='#1f77b4'/><circle cx='265.1428571428571' cy='248.0' r='3' fill='#1f77b4'/><circle cx='393.0285714285714' cy='248.0' r='3' fill='#1f77b4'/><circle cx='648.8' cy='248.0' r='3' fill='#1f77b4'/><polyline points='52.0,203.17795039224572 137.25714285714287,244.58932667946542 265.1428571428571,244.40629164164943 393.0285714285714,247.6239038272387 648.8,245.87982975590813' fill='none' stroke='#d62728' stroke-width='1.8'/><circle cx='52.0' cy='203.17795039224572' r='3' fill='#d62728'/><circle cx='137.25714285714287' cy='244.58932667946542' r='3' fill='#d62728'/><circle cx='265.1428571428571' cy='244.40629164164943' r='3' fill='#d62728'/><circle cx='393.0285714285714' cy='247.6239038272387' r='3' fill='#d62728'/><circle cx='648.8' cy='245.87982975590813' r='3' fill='#d62728'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#1f77b4' stroke-width='2.4'/><text x='521' y='44.0'>primary_contact_time</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#d62728' stroke-width='2.4'/><text x='521' y='58.0'>CoR</text><text x='340.0' y='292' text-anchor='middle'>M</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>% change from previous M</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Integrated metrics vs M</text></svg>")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>40.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.002</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>75.0</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>0.117</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>110.0</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>0.233</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>145.0</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>0.348</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>180.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>0.463</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,36.39999999999998 137.25714285714287,158.4419426808039 265.1428571428571,210.3748094222112 393.0285714285714,247.2404515632464 648.8,240.33919127428112' fill='none' stroke='#2ca02c' stroke-width='1.8'/><circle cx='52.0' cy='36.39999999999998' r='3' fill='#2ca02c'/><circle cx='137.25714285714287' cy='158.4419426808039' r='3' fill='#2ca02c'/><circle cx='265.1428571428571' cy='210.3748094222112' r='3' fill='#2ca02c'/><circle cx='393.0285714285714' cy='247.2404515632464' r='3' fill='#2ca02c'/><circle cx='648.8' cy='240.33919127428112' r='3' fill='#2ca02c'/><polyline points='52.0,194.31381175533573 137.25714285714287,189.57248178465454 265.1428571428571,230.75588857126104 393.0285714285714,248.0 648.8,243.60118671033302' fill='none' stroke='#9467bd' stroke-width='1.8'/><circle cx='52.0' cy='194.31381175533573' r='3' fill='#9467bd'/><circle cx='137.25714285714287' cy='189.57248178465454' r='3' fill='#9467bd'/><circle cx='265.1428571428571' cy='230.75588857126104' r='3' fill='#9467bd'/><circle cx='393.0285714285714' cy='248.0' r='3' fill='#9467bd'/><circle cx='648.8' cy='243.60118671033302' r='3' fill='#9467bd'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#2ca02c' stroke-width='2.4'/><text x='521' y='44.0'>c_1 (bath moment)</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#9467bd' stroke-width='2.4'/><text x='521' y='58.0'>b_2 (droplet moment)</text><text x='340.0' y='292' text-anchor='middle'>M</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>% change from previous M</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Pressure moments vs M</text></svg>")

Verdicts (M-sweep):
  primary_contact_time: converged by index 2 (value 3.6293), consecutive deviations 0.554%, 0%
  CoR: converged by index 2 (value 0.29959), consecutive deviations 0.117%, 0.00893%


  c_1: converged by index 2 (value 0.086437), consecutive deviations 0.463%, 0.197%
  b_2: converged by index 2 (value 0.096166), consecutive deviations 0.119%, 0.129%


## `L`: droplet (Legendre) modes

`M=120` held fixed so the droplet truncation is isolated.

In [5]:
Ls = (20, 40, 60, 90, 120, 180)
results_L = [run_case(M=120, L=l, N=3) for l in Ls]
display(svgplot([(x=collect(Ls[2:end]), y=reldev(results_L, r -> r.primary)[2:end], color="#1f77b4", label="primary_contact_time"),
                  (x=collect(Ls[2:end]), y=reldev(results_L, r -> r.cor)[2:end], color="#d62728", label="CoR")];
                 xlabel="L", ylabel="% change from previous L", title="Integrated metrics vs L"))
display(svgplot([(x=collect(Ls[2:end]), y=reldev(results_L, r -> r.c1)[2:end], color="#2ca02c", label="c_1 (bath moment)"),
                  (x=collect(Ls[2:end]), y=reldev(results_L, r -> r.b2)[2:end], color="#9467bd", label="b_2 (droplet moment)")];
                 xlabel="L", ylabel="% change from previous L", title="Pressure moments vs L"))
println("Verdicts (L-sweep):")
verdict("primary_contact_time", results_L, r -> r.primary)
verdict("CoR", results_L, r -> r.cor)
verdict("c_1", results_L, r -> r.c1)
verdict("b_2", results_L, r -> r.b2)

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>40.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.0</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>75.0</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>0.00545</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>110.0</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>0.0109</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>145.0</text><line x1='52' y1='89.29999999999995' x2='47' y2='89.29999999999995' stroke='black'/><text x='43' y='93.29999999999995' text-anchor='end'>0.0164</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>180.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>0.0218</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,248.0 137.25714285714287,248.0 265.1428571428571,248.0 393.0285714285714,248.0 648.8,248.0' fill='none' stroke='#1f77b4' stroke-width='1.8'/><circle cx='52.0' cy='248.0' r='3' fill='#1f77b4'/><circle cx='137.25714285714287' cy='248.0' r='3' fill='#1f77b4'/><circle cx='265.1428571428571' cy='248.0' r='3' fill='#1f77b4'/><circle cx='393.0285714285714' cy='248.0' r='3' fill='#1f77b4'/><circle cx='648.8' cy='248.0' r='3' fill='#1f77b4'/><polyline points='52.0,244.61680258110314 137.25714285714287,36.39999999999998 265.1428571428571,151.19508779187925 393.0285714285714,244.89394937398265 648.8,247.15223275324948' fill='none' stroke='#d62728' stroke-width='1.8'/><circle cx='52.0' cy='244.61680258110314' r='3' fill='#d62728'/><circle cx='137.25714285714287' cy='36.39999999999998' r='3' fill='#d62728'/><circle cx='265.1428571428571' cy='151.19508779187925' r='3' fill='#d62728'/><circle cx='393.0285714285714' cy='244.89394937398265' r='3' fill='#d62728'/><circle cx='648.8' cy='247.15223275324948' r='3' fill='#d62728'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#1f77b4' stroke-width='2.4'/><text x='521' y='44.0'>primary_contact_time</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#d62728' stroke-width='2.4'/><text x='521' y='58.0'>CoR</text><text x='340.0' y='292' text-anchor='middle'>L</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>% change from previous L</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Integrated metrics vs L</text></svg>")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>40.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.0011</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>75.0</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>0.0263</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>110.0</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>0.0516</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>145.0</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>0.0768</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>180.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>0.102</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,36.39999999999998 137.25714285714287,227.045189407701 265.1428571428571,243.4544395323736 393.0285714285714,208.5961228650514 648.8,248.0' fill='none' stroke='#2ca02c' stroke-width='1.8'/><circle cx='52.0' cy='36.39999999999998' r='3' fill='#2ca02c'/><circle cx='137.25714285714287' cy='227.045189407701' r='3' fill='#2ca02c'/><circle cx='265.1428571428571' cy='243.4544395323736' r='3' fill='#2ca02c'/><circle cx='393.0285714285714' cy='208.5961228650514' r='3' fill='#2ca02c'/><circle cx='648.8' cy='248.0' r='3' fill='#2ca02c'/><polyline points='52.0,190.6538156981656 137.25714285714287,237.1995408102864 265.1428571428571,243.50459595572423 393.0285714285714,224.95837244943507 648.8,221.96121893413712' fill='none' stroke='#9467bd' stroke-width='1.8'/><circle cx='52.0' cy='190.6538156981656' r='3' fill='#9467bd'/><circle cx='137.25714285714287' cy='237.1995408102864' r='3' fill='#9467bd'/><circle cx='265.1428571428571' cy='243.50459595572423' r='3' fill='#9467bd'/><circle cx='393.0285714285714' cy='224.95837244943507' r='3' fill='#9467bd'/><circle cx='648.8' cy='221.96121893413712' r='3' fill='#9467bd'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#2ca02c' stroke-width='2.4'/><text x='521' y='44.0'>c_1 (bath moment)</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#9467bd' stroke-width='2.4'/><text x='521' y='58.0'>b_2 (droplet moment)</text><text x='340.0' y='292' text-anchor='middle'>L</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>% change from previous L</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Pressure moments vs L</text></svg>")

Verdicts (L-sweep):
  primary_contact_time: converged by index 2 (value 3.6293), consecutive deviations 0%, 0%
  CoR: converged by index 2 (value 0.29963), consecutive deviations 0.000349%, 0.0218%


  c_1: converged by index 2 (value 0.086562), consecutive deviations 0.102%, 0.0111%
  b_2: converged by index 2 (value 0.096271), consecutive deviations 0.0285%, 0.00626%


## `N`: pressure modes, and the Newton Jacobian's conditioning

`M=L=120` held fixed. This is the one truncation the design doc already suspected would not
matter for the integrated metrics -- checked properly here against the moments too -- and the
one where solver conditioning genuinely changes with the parameter being swept, so both are
plotted.

In [6]:
Ns = (1, 2, 3, 4, 6, 8)
results_N = [run_case(M=120, L=120, N=n) for n in Ns]
display(svgplot([(x=collect(Ns[2:end]), y=reldev(results_N, r -> r.primary)[2:end], color="#1f77b4", label="primary_contact_time"),
                  (x=collect(Ns[2:end]), y=reldev(results_N, r -> r.cor)[2:end], color="#d62728", label="CoR")];
                 xlabel="N", ylabel="% change from previous N", title="Integrated metrics vs N"))
display(svgplot([(x=collect(Ns[2:end]), y=reldev(results_N, r -> r.c1)[2:end], color="#2ca02c", label="c_1 (bath moment)"),
                  (x=collect(Ns[2:end]), y=reldev(results_N, r -> r.b2)[2:end], color="#9467bd", label="b_2 (droplet moment)")];
                 xlabel="N", ylabel="% change from previous N", title="Pressure moments vs N"))
display(svgplot([(x=collect(Ns), y=[log10(r.condJ) for r in results_N], color="#8c564b", label="log10(cond J)")];
                 xlabel="N", ylabel="log10(cond J) at peak penetration", title="Newton Jacobian conditioning vs N"))
println("Verdicts (N-sweep):")
verdict("primary_contact_time", results_N, r -> r.primary)
verdict("CoR", results_N, r -> r.cor)
verdict("c_1", results_N, r -> r.c1)
verdict("b_2", results_N, r -> r.b2)
@printf("  cond(J): %.3g -> %.3g over N=%d..%d\n", results_N[1].condJ, results_N[end].condJ, Ns[1], Ns[end])

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>2.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.0</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>3.5</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>0.00313</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>5.0</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>0.00625</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>6.5</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>0.00938</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>8.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>0.0125</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,248.0 151.46666666666664,248.0 250.9333333333333,248.0 449.8666666666666,248.0 648.8,248.0' fill='none' stroke='#1f77b4' stroke-width='1.8'/><circle cx='52.0' cy='248.0' r='3' fill='#1f77b4'/><circle cx='151.46666666666664' cy='248.0' r='3' fill='#1f77b4'/><circle cx='250.9333333333333' cy='248.0' r='3' fill='#1f77b4'/><circle cx='449.8666666666666' cy='248.0' r='3' fill='#1f77b4'/><circle cx='648.8' cy='248.0' r='3' fill='#1f77b4'/><polyline points='52.0,36.39999999999998 151.46666666666664,247.12022248690215 250.9333333333333,240.41157497027842 449.8666666666666,236.14034009476873 648.8,233.34301804588188' fill='none' stroke='#d62728' stroke-width='1.8'/><circle cx='52.0' cy='36.39999999999998' r='3' fill='#d62728'/><circle cx='151.46666666666664' cy='247.12022248690215' r='3' fill='#d62728'/><circle cx='250.9333333333333' cy='240.41157497027842' r='3' fill='#d62728'/><circle cx='449.8666666666666' cy='236.14034009476873' r='3' fill='#d62728'/><circle cx='648.8' cy='233.34301804588188' r='3' fill='#d62728'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#1f77b4' stroke-width='2.4'/><text x='521' y='44.0'>primary_contact_time</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#d62728' stroke-width='2.4'/><text x='521' y='58.0'>CoR</text><text x='340.0' y='292' text-anchor='middle'>N</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>% change from previous N</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Integrated metrics vs N</text></svg>")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>2.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.00401</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>3.5</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>0.112</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>5.0</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>0.219</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>6.5</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>0.327</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>8.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>0.435</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,243.78832390258924 151.46666666666664,240.51488983786004 250.9333333333333,248.0 449.8666666666666,233.41206037422305 648.8,245.348928480438' fill='none' stroke='#2ca02c' stroke-width='1.8'/><circle cx='52.0' cy='243.78832390258924' r='3' fill='#2ca02c'/><circle cx='151.46666666666664' cy='240.51488983786004' r='3' fill='#2ca02c'/><circle cx='250.9333333333333' cy='248.0' r='3' fill='#2ca02c'/><circle cx='449.8666666666666' cy='233.41206037422305' r='3' fill='#2ca02c'/><circle cx='648.8' cy='245.348928480438' r='3' fill='#2ca02c'/><polyline points='52.0,36.39999999999998 151.46666666666664,243.10963067454045 250.9333333333333,247.50338733240736 449.8666666666666,244.57500467586158 648.8,247.39534527069995' fill='none' stroke='#9467bd' stroke-width='1.8'/><circle cx='52.0' cy='36.39999999999998' r='3' fill='#9467bd'/><circle cx='151.46666666666664' cy='243.10963067454045' r='3' fill='#9467bd'/><circle cx='250.9333333333333' cy='247.50338733240736' r='3' fill='#9467bd'/><circle cx='449.8666666666666' cy='244.57500467586158' r='3' fill='#9467bd'/><circle cx='648.8' cy='247.39534527069995' r='3' fill='#9467bd'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#2ca02c' stroke-width='2.4'/><text x='521' y='44.0'>c_1 (bath moment)</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#9467bd' stroke-width='2.4'/><text x='521' y='58.0'>b_2 (droplet moment)</text><text x='340.0' y='292' text-anchor='middle'>N</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>% change from previous N</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Pressure moments vs N</text></svg>")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>1.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.332</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>2.75</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>0.65</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>4.5</text><line x1='52' y1='142.20000000000002' x2='47' y2='142.20000000000002' stroke='black'/><text x='43' y='146.20000000000002' text-anchor='end'>0.968</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>6.25</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>1.29</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>8.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>1.6</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,248.0 137.25714285714287,238.51906090086658 222.5142857142857,223.6876012347913 307.77142857142854,192.62489610025503 478.2857142857143,116.36036622596208 648.8,36.39999999999998' fill='none' stroke='#8c564b' stroke-width='1.8'/><circle cx='52.0' cy='248.0' r='3' fill='#8c564b'/><circle cx='137.25714285714287' cy='238.51906090086658' r='3' fill='#8c564b'/><circle cx='222.5142857142857' cy='223.6876012347913' r='3' fill='#8c564b'/><circle cx='307.77142857142854' cy='192.62489610025503' r='3' fill='#8c564b'/><circle cx='478.2857142857143' cy='116.36036622596208' r='3' fill='#8c564b'/><circle cx='648.8' cy='36.39999999999998' r='3' fill='#8c564b'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#8c564b' stroke-width='2.4'/><text x='521' y='44.0'>log10(cond J)</text><text x='340.0' y='292' text-anchor='middle'>N</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>log10(cond J) at peak penetration</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Newton Jacobian conditioning vs N</text></svg>")

Verdicts (N-sweep):
  primary_contact_time: converged by index 2 (value 3.6293), consecutive deviations 0%, 0%
  CoR: converged by index 2 (value 0.29953), consecutive deviations 0.0125%, 5.2e-05%


  c_1: converged by index 2 (value 0.086549), consecutive deviations 0.0126%, 0.0192%
  b_2: converged by index 2 (value 0.096264), consecutive deviations 0.435%, 0.014%
  cond(J): 2.15 -> 40.1 over N=1..8


## Joint `M=L` cross-check

The design doc's own history includes a withdrawn resolvable-rank law whose supporting study
set `M=L` throughout and, as a result, could not tell a genuine bath effect from a genuine
droplet effect -- they were collinear by construction. The decoupled sweeps above avoid that
mistake. This section runs the check in the other direction: does refining `M` and `L`
*together* reveal anything the decoupled sweeps missed, or do the two agree?

In [7]:
MLs = (20, 40, 80, 160)
results_ML = [run_case(M=v, L=v, N=3) for v in MLs]
display(svgplot([(x=collect(MLs[2:end]), y=reldev(results_ML, r -> r.primary)[2:end], color="#1f77b4", label="primary_contact_time"),
                  (x=collect(MLs[2:end]), y=reldev(results_ML, r -> r.cor)[2:end], color="#d62728", label="CoR")];
                 xlabel="M=L", ylabel="% change from previous M=L", title="Integrated metrics, M=L doubling"))
display(svgplot([(x=collect(MLs[2:end]), y=reldev(results_ML, r -> r.c1)[2:end], color="#2ca02c", label="c_1 (bath moment)"),
                  (x=collect(MLs[2:end]), y=reldev(results_ML, r -> r.b2)[2:end], color="#9467bd", label="b_2 (droplet moment)")];
                 xlabel="M=L", ylabel="% change from previous M=L", title="Pressure moments, M=L doubling"))
println("Verdicts (joint M=L sweep):")
verdict("primary_contact_time", results_ML, r -> r.primary)
verdict("CoR", results_ML, r -> r.cor)
verdict("c_1", results_ML, r -> r.c1)
verdict("b_2", results_ML, r -> r.b2)

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>40.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.0</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>70.0</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>0.52</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>100.0</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>1.04</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>130.0</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>1.56</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>160.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>2.08</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,36.39999999999998 250.9333333333333,248.0 648.8,248.0' fill='none' stroke='#1f77b4' stroke-width='1.8'/><circle cx='52.0' cy='36.39999999999998' r='3' fill='#1f77b4'/><circle cx='250.9333333333333' cy='248.0' r='3' fill='#1f77b4'/><circle cx='648.8' cy='248.0' r='3' fill='#1f77b4'/><polyline points='52.0,213.14999036432008 250.9333333333333,244.39124911253748 648.8,244.89842116456632' fill='none' stroke='#d62728' stroke-width='1.8'/><circle cx='52.0' cy='213.14999036432008' r='3' fill='#d62728'/><circle cx='250.9333333333333' cy='244.39124911253748' r='3' fill='#d62728'/><circle cx='648.8' cy='244.89842116456632' r='3' fill='#d62728'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#1f77b4' stroke-width='2.4'/><text x='521' y='44.0'>primary_contact_time</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#d62728' stroke-width='2.4'/><text x='521' y='58.0'>CoR</text><text x='340.0' y='292' text-anchor='middle'>M=L</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>% change from previous M=L</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Integrated metrics, M=L doubling</text></svg>")

HTML("<svg xmlns='http://www.w3.org/2000/svg' width='680' height='300' font-family='sans-serif' font-size='12'><rect width='680' height='300' fill='white'/><line x1='52.0' y1='248' x2='52.0' y2='253' stroke='black'/><text x='52.0' y='266' text-anchor='middle'>40.0</text><line x1='52' y1='248.0' x2='47' y2='248.0' stroke='black'/><text x='43' y='252.0' text-anchor='end'>0.0069</text><line x1='201.2' y1='248' x2='201.2' y2='253' stroke='black'/><text x='201.2' y='266' text-anchor='middle'>70.0</text><line x1='52' y1='195.1' x2='47' y2='195.1' stroke='black'/><text x='43' y='199.1' text-anchor='end'>0.618</text><line x1='350.4' y1='248' x2='350.4' y2='253' stroke='black'/><text x='350.4' y='266' text-anchor='middle'>100.0</text><line x1='52' y1='142.2' x2='47' y2='142.2' stroke='black'/><text x='43' y='146.2' text-anchor='end'>1.23</text><line x1='499.59999999999997' y1='248' x2='499.59999999999997' y2='253' stroke='black'/><text x='499.59999999999997' y='266' text-anchor='middle'>130.0</text><line x1='52' y1='89.29999999999998' x2='47' y2='89.29999999999998' stroke='black'/><text x='43' y='93.29999999999998' text-anchor='end'>1.84</text><line x1='648.8' y1='248' x2='648.8' y2='253' stroke='black'/><text x='648.8' y='266' text-anchor='middle'>160.0</text><line x1='52' y1='36.39999999999998' x2='47' y2='36.39999999999998' stroke='black'/><text x='43' y='40.39999999999998' text-anchor='end'>2.45</text><line x1='52' y1='248' x2='648.8' y2='248' stroke='black'/><line x1='52' y1='248' x2='52' y2='26.0' stroke='black'/><polyline points='52.0,36.39999999999998 250.9333333333333,242.84720710885972 648.8,248.0' fill='none' stroke='#2ca02c' stroke-width='1.8'/><circle cx='52.0' cy='36.39999999999998' r='3' fill='#2ca02c'/><circle cx='250.9333333333333' cy='242.84720710885972' r='3' fill='#2ca02c'/><circle cx='648.8' cy='248.0' r='3' fill='#2ca02c'/><polyline points='52.0,176.36856446152456 250.9333333333333,243.0771438787817 648.8,247.977860523303' fill='none' stroke='#9467bd' stroke-width='1.8'/><circle cx='52.0' cy='176.36856446152456' r='3' fill='#9467bd'/><circle cx='250.9333333333333' cy='243.0771438787817' r='3' fill='#9467bd'/><circle cx='648.8' cy='247.977860523303' r='3' fill='#9467bd'/><line x1='490' y1='40.0' x2='515' y2='40.0' stroke='#2ca02c' stroke-width='2.4'/><text x='521' y='44.0'>c_1 (bath moment)</text><line x1='490' y1='54.0' x2='515' y2='54.0' stroke='#9467bd' stroke-width='2.4'/><text x='521' y='58.0'>b_2 (droplet moment)</text><text x='340.0' y='292' text-anchor='middle'>M=L</text><text x='14' y='150.0' text-anchor='middle' transform='rotate(-90 14 150.0)'>% change from previous M=L</text><text x='340.0' y='18' text-anchor='middle' font-size='14'>Pressure moments, M=L doubling</text></svg>")

Verdicts (joint M=L sweep):
  primary_contact_time: converged by index 3 (value 3.6293), consecutive deviations 0%, 0%
  CoR: converged by index 2 (value 0.2997), consecutive deviations 0.343%, 0.0355%


  c_1: converged by index 3 (value 0.086527), consecutive deviations 0.0664%, 0.0069%
  b_2: converged by index 2 (value 0.096188), consecutive deviations 0.835%, 0.0638%


## Verdict

(filled in after the runs above; see the printed convergence indices)

In [8]:
"""Smallest swept value at or after which `getter` stays within `tol`% of its predecessor
for two consecutive refinements, or `nothing` if that never happens in the range tested."""
function converged_value(paramvals, results, getter; tol=1.0)
    d = reldev(results, getter)
    ok = [i >= 3 && d[i] < tol && d[i-1] < tol for i in eachindex(d)]
    idx = findfirst(ok)
    return idx === nothing ? nothing : paramvals[idx-1]
end

function required(paramvals, results, floor)
    reqs = [converged_value(paramvals, results, g) for g in (r -> r.primary, r -> r.cor, r -> r.c1, r -> r.b2)]
    any(isnothing, reqs) && return (nothing, reqs)
    return (max(floor, maximum(reqs)), reqs)
end

println("="^70)
println("DECISIVE VERDICT")
println("="^70)
println()
println("M-sweep (L=120, N=3):"); verdict("primary_contact_time", results_M, r -> r.primary); verdict("CoR", results_M, r -> r.cor); verdict("c_1", results_M, r -> r.c1); verdict("b_2", results_M, r -> r.b2)
println()
println("L-sweep (M=120, N=3):"); verdict("primary_contact_time", results_L, r -> r.primary); verdict("CoR", results_L, r -> r.cor); verdict("c_1", results_L, r -> r.c1); verdict("b_2", results_L, r -> r.b2)
println()
println("N-sweep (M=L=120):"); verdict("primary_contact_time", results_N, r -> r.primary); verdict("CoR", results_N, r -> r.cor); verdict("c_1", results_N, r -> r.c1); verdict("b_2", results_N, r -> r.b2)
println()
println("Joint M=L:"); verdict("primary_contact_time", results_ML, r -> r.primary); verdict("CoR", results_ML, r -> r.cor); verdict("c_1", results_ML, r -> r.c1); verdict("b_2", results_ML, r -> r.b2)

println()
println("-"^70)
println("Synthesis: production must satisfy the DECOUPLED requirement (each axis refined")
println("alone, the other held generously at 120) AND the JOINT requirement (both refined")
println("together, the realistic case) simultaneously -- take the larger of the two.")
M_decoupled, _ = required(collect(Ms), results_M, 20)
L_decoupled, _ = required(collect(Ls), results_L, 20)
N_req, _ = required(collect(Ns), results_N, 1)
ML_joint, _ = required(collect(MLs), results_ML, 20)

if M_decoupled === nothing || L_decoupled === nothing || N_req === nothing || ML_joint === nothing
    println("At least one quantity did not converge to 1% within the range tested -- see the")
    println("individual verdicts above for which one, and extend that sweep's range.")
else
    M_rec = max(M_decoupled, ML_joint, 60)
    L_rec = max(L_decoupled, ML_joint, 60)
    N_rec = max(N_req, 3)
    @printf("  M: decoupled requires >= %d, joint (M=L) requires >= %d -> recommend M = %d\n", M_decoupled, ML_joint, M_rec)
    @printf("  L: decoupled requires >= %d, joint (M=L) requires >= %d -> recommend L = %d\n", L_decoupled, ML_joint, L_rec)
    @printf("  N: requires >= %d (cond(J) stays order-unity through N=3-4) -> recommend N = %d\n", N_req, N_rec)
    println()
    if ML_joint > 60
        @printf("RECOMMENDED PRODUCTION: (M, L, N) = (%d, %d, %d).\n", M_rec, L_rec, N_rec)
        println("This RAISES production from the current M=L=60: decoupling M and L (each refined")
        println("alone against a generous L=120 or M=120) converges earlier than the JOINT M=L sweep")
        println("does -- exactly the confound `derivations/`'s own history warns an M=L-only study")
        println("cannot see. Refining both together is the realistic case, and it needs more.")
    else
        @printf("RECOMMENDED PRODUCTION: (M, L, N) = (%d, %d, %d) -- current production already meets it.\n", M_rec, L_rec, N_rec)
    end
end

DECISIVE VERDICT

M-sweep (L=120, N=3):
  primary_contact_time: converged by index 2 (value 3.6293), consecutive deviations 0.554%, 0%


  CoR: converged by index 2 (value 0.29959), consecutive deviations 0.117%, 0.00893%
  c_1: converged by index 2 (value 0.086437), consecutive deviations 0.463%, 0.197%
  b_2: converged by index 2 (value 0.096166), consecutive deviations 0.119%, 0.129%



L-sweep (M=120, N=3):
  primary_contact_time: converged by index 2 (value 3.6293), consecutive deviations 0%, 0%
  CoR: converged by index 2 (value 0.29963), consecutive deviations 0.000349%, 0.0218%


  c_1: converged by index 2 (value 0.086562), consecutive deviations 0.102%, 0.0111%
  b_2: converged by index 2 (value 0.096271), consecutive deviations 0.0285%, 0.00626%

N-sweep (M=L=120):
  primary_contact_time: converged by index 2 (value 3.6293), consecutive deviations 0%, 0%


  CoR: converged by index 2 (value 0.29953), consecutive deviations 0.0125%, 5.2e-05%
  c_1: converged by index 2 (value 0.086549), consecutive deviations 0.0126%, 0.0192%
  b_2: converged by index 2 (value 0.096264), consecutive deviations 0.435%, 0.014%



Joint M=L:
  primary_contact_time: converged by index 3 (value 3.6293), consecutive deviations 0%, 0%
  CoR: converged by index 2 (value 0.2997), consecutive deviations 0.343%, 0.0355%


  c_1: converged by index 3 (value 0.086527), consecutive deviations 0.0664%, 0.0069%
  b_2: converged by index 2 (value 0.096188), consecutive deviations 0.835%, 0.0638%

----------------------------------------------------------------------
Synthesis: production must satisfy the DECOUPLED requirement (each axis refined
alone, the other held generously at 120) AND the JOINT requirement (both refined
together, the realistic case) simultaneously -- take the larger of the two.
  M: decoupled requires >= 40, joint (M=L) requires >= 80 -> recommend M = 80


  L: decoupled requires >= 40, joint (M=L) requires >= 80 -> recommend L = 80
  N: requires >= 2 (cond(J) stays order-unity through N=3-4) -> recommend N = 3

RECOMMENDED PRODUCTION: (M, L, N) = (80, 80, 3).
This RAISES production from the current M=L=60: decoupling M and L (each refined
alone against a generous L=120 or M=120) converges earlier than the JOINT M=L sweep
does -- exactly the confound `derivations/`'s own history warns an M=L-only study
cannot see. Refining both together is the realistic case, and it needs more.
